In [1]:
import math
import torch
import torch.nn as nn

torch.manual_seed(42)

### Positional Encoding class

In [2]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer("pe", pe.unsqueeze(0))

    
    def forward(self, x):
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len]
        return x

In [3]:
batch_size = 1
seq_len = 4
d_model = 8

x = torch.randn(batch_size, seq_len, d_model)
print("Original token embedding shape:", x.shape)
print(x)

Original token embedding shape: torch.Size([1, 4, 8])
tensor([[[ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784, -1.2345, -0.0431,
          -1.6047],
         [-0.7521,  1.6487, -0.3925, -1.4036, -0.7279, -0.5594, -0.7688,
           0.7624],
         [ 1.6423, -0.1596, -0.4974,  0.4396, -0.7581,  1.0783,  0.8008,
           1.6806],
         [ 1.2791,  1.2964,  0.6105,  1.3347, -0.2316,  0.0418, -0.2516,
           0.8599]]])


**Apply positional encoding**

In [4]:
pos_encoder = SinusoidalPositionalEncoding(d_model=d_model, max_len=10)

x_with_pe = pos_encoder(x)

print("After positional encoding shape:", x_with_pe.shape)
print(x_with_pe)

After positional encoding shape: torch.Size([1, 4, 8])
tensor([[[ 1.9269,  2.4873,  0.9007, -1.1055,  0.6784, -0.2345, -0.0431,
          -0.6047],
         [ 0.0893,  2.1890, -0.2926, -0.4086, -0.7179,  0.4405, -0.7678,
           1.7624],
         [ 2.5516, -0.5757, -0.2987,  1.4197, -0.7381,  2.0781,  0.8028,
           2.6806],
         [ 1.4202,  0.3064,  0.9060,  2.2901, -0.2016,  1.0413, -0.2486,
           1.8599]]])


In [5]:
print("Positional Encoding Table Shape:", pos_encoder.pe.shape)
print(pos_encoder.pe[0, :4])

Positional Encoding Table Shape: torch.Size([1, 10, 8])
tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  1.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  9.9833e-02,  9.9500e-01,  9.9998e-03,
          9.9995e-01,  1.0000e-03,  1.0000e+00],
        [ 9.0930e-01, -4.1615e-01,  1.9867e-01,  9.8007e-01,  1.9999e-02,
          9.9980e-01,  2.0000e-03,  1.0000e+00],
        [ 1.4112e-01, -9.8999e-01,  2.9552e-01,  9.5534e-01,  2.9995e-02,
          9.9955e-01,  3.0000e-03,  1.0000e+00]])


In [6]:
print("Position 0 encoding:", pos_encoder.pe[0, 0])
print("Position 1 encoding:", pos_encoder.pe[0, 1])
print("Position 2 encoding:", pos_encoder.pe[0, 2])

Position 0 encoding: tensor([0., 1., 0., 1., 0., 1., 0., 1.])
Position 1 encoding: tensor([0.8415, 0.5403, 0.0998, 0.9950, 0.0100, 0.9999, 0.0010, 1.0000])
Position 2 encoding: tensor([ 0.9093, -0.4161,  0.1987,  0.9801,  0.0200,  0.9998,  0.0020,  1.0000])


**Token embedding + position encoding manually**

In [7]:
token_embedding = x[0, 0]
position_encoding = pos_encoder.pe[0, 0]

final_rep = token_embedding + position_encoding

print("Token embedding:", token_embedding)
print("Position encoding:", position_encoding)
print("Final representation:", final_rep)

Token embedding: tensor([ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784, -1.2345, -0.0431, -1.6047])
Position encoding: tensor([0., 1., 0., 1., 0., 1., 0., 1.])
Final representation: tensor([ 1.9269,  2.4873,  0.9007, -1.1055,  0.6784, -0.2345, -0.0431, -0.6047])
